# 15 · Grid search (w, α) trên val (ticket #25)

Chọn cấu hình cho bảng kết quả chính (spec §6) với mô hình gu app dùng, chọn như notebook 14: NDCG@10 val cao nhất trong `{model}_params.json` (hiện là **ALS**, vector bài và decay đã chọn ở #14). Notebook in tên mô hình ở dòng đầu.
- Điểm mix theo luật: 21 bộ trọng số w = (BPM, tông, energy), bước 0,2, tổng bằng 1, × α ∈ {0; 0,1; …; 1} = 231 cấu hình.
- Điểm mix học (#23): chỉ chọn α (11 cấu hình).
- Độ chính xác (HitRate@10, NDCG@10, MRR) và độ mượt (chuỗi tham lam 10 bài, #18) trên **cùng 2.000 phiên val** cố định (`random_seed` 42, cùng mẫu đo độ mượt ở #19).
- Quy tắc chọn: tỉ lệ cặp mix được cao nhất mà NDCG@10 giảm ≤ 10% so với α = 1. Nếu chỉ α = 1 đạt ngưỡng, notebook in **CẢNH BÁO**: báo cả nhóm trước khi chỉnh.

Code: `src/eval/grid.py` (có test: độ chính xác vector hóa và chuỗi sinh chung đều trùng với cách tính từng phiên của #19).

**Input** (Add Input), giống notebook 14: `titeohihi/mixnext-splited`, `mixnext-embeddings` (ALS), `mixnext-item2vec`, `mixnext-mix-learned`. Không thêm `mixnext-lastfm` (cũng có `tracks.parquet`, script sẽ báo trùng).

**Output** (`/kaggle/working/out/`): `grid.csv` (242 dòng), `mix_params.json` (định dạng `artifacts/`).

Bật **Internet** để clone repo. Chỉ cần CPU; ước tính 1–2 giờ (in thời gian từng α).

In [ ]:
# Bootstrap: lấy code từ GitHub. Đang làm trên nhánh b; khi #25 đã merge thì đổi thành "main".
BRANCH = "b"
REPO = "/tmp/MusicRecSys"
!rm -rf {REPO} && git clone -q --depth 1 -b {BRANCH} https://github.com/yashita23/MusicRecSys {REPO}
import sys; sys.path.insert(0, REPO)

In [ ]:
from pathlib import Path
from src.eval.grid import main

OUT = Path("/kaggle/working/out")
table = main(["--input", "/kaggle/input", "--out", str(OUT)])   # tìm file theo tên dưới /kaggle/input; báo lỗi nếu thiếu hoặc trùng

In [ ]:
import pandas as pd
pd.set_option("display.width", 200)
grid = pd.read_csv(OUT / "grid.csv")
cols = ["mix", "w_bpm", "w_key", "w_energy", "alpha", "ndcg", "hit_rate", "mrr", "mixable_pct", "key_compatible_pct", "bpm_delta_pct", "energy_gap"]
rule, learned = grid[grid["mix"] == "rule"], grid[grid["mix"] == "learned"]
threshold = 0.9 * rule.loc[rule["alpha"] == 1.0, "ndcg"].iloc[0]
print(f"Ngưỡng NDCG@10 (luật) = 0,9 × NDCG ở α = 1 = {threshold:.4f}")

print("\nCấu hình được chọn:")
print(grid.loc[grid["chosen"], cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nĐạt ngưỡng, top 10 theo tỉ lệ cặp mix được (luật):")
ok = rule[rule["ndcg"] >= threshold - 1e-12].sort_values(["mixable_pct", "alpha"], ascending=False)
print(ok[cols].head(10).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nMỗi α: bộ w mượt nhất trong các bộ đạt ngưỡng (luật):")
best = ok.sort_values("mixable_pct", ascending=False).groupby("alpha", sort=True).head(1).sort_values("alpha", ascending=False)
print(best[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\nMix học theo α:")
print(learned[cols].to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("\nmix_params.json:")
print((OUT / "mix_params.json").read_text(encoding="utf-8"))

## Đăng output

1. **Save Version → Save & Run All (Commit)**.
2. Tab Output → **New Dataset** tên `mixnext-grid`, để **Private**, chỉ giữ `out/` (`grid.csv`, `mix_params.json`).
3. Notebook 14 (#20): Add Input thêm `mixnext-grid` rồi chạy lại, để `artifacts/mix_params.json` là cấu hình vừa chọn (app dùng α và w trong file này).
4. Chép `grid.csv` vào `reports/grid.csv` của repo (#25).

## Kết quả (Kaggle, CPU, ngày …)

Điền sau khi chạy: ngưỡng NDCG, cấu hình được chọn cho luật (w, α) và mix học (α), so với α = 1 và với w mặc định 0,4/0,4/0,2, thời gian chạy.